# VoiceHub Dia workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/tts_workflow.ipynb)

A compact baseline → data → one-step fine-tune → export → reload workflow. Keep all run flags off until the preceding check passes.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("voicehub") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "voicehub[training] @ git+https://github.com/kadirnar/voicehub.git@main",
    ])

## 1. Configure

In [ ]:
from pathlib import Path

RUN_INFERENCE = False
RUN_TRAINING = False
RUN_POST_TRAINING_INFERENCE = False

MODEL_TYPE = "dia"
BASE_MODEL = "nari-labs/Dia-1.6B-0626"
DEVICE = "cuda"
DATA_ROOT = Path("data/dia")
MANIFEST_PATH = DATA_ROOT / "manifest.jsonl"
OUTPUT_DIR = Path("artifacts/dia-workflow")

EVALUATION_TEXT = (
    "[S1] VoiceHub keeps the complete speech workflow explicit and easy to inspect. "
    "[S2] We will compare the same long prompt before and after training, verify that "
    "each generated sample lasts at least ten seconds, and record every setting needed "
    "to reproduce the result. We will also listen for stable volume, clear word endings, "
    "natural pauses, and consistent speaker identity throughout the complete recording."
)

## 2. Inspect support

In [ ]:
from voicehub.training.arguments import TrainingArguments
from voicehub.training import get_training_spec, get_tts_dataset_spec

training_spec = get_training_spec(MODEL_TYPE)
dataset_spec = get_tts_dataset_spec(MODEL_TYPE)
print(training_spec.support.value, training_spec.family_name)
print(dataset_spec.architecture.value, dataset_spec.readiness.value)

## 3. Generate the baseline

The waveform check enforces the requested ten-second minimum.

In [ ]:
if RUN_INFERENCE:
    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    baseline_model = AutoModelForTextToSpeech.from_pretrained(
        BASE_MODEL, model_type=MODEL_TYPE, device=DEVICE, lazy_load=True
    )
    baseline_output = baseline_model.generate(
        EVALUATION_TEXT,
        generation_config=TTSGenerationConfig(
            seed=42, max_new_tokens=2048, output_file=OUTPUT_DIR / "baseline.wav"
        ),
    )
    baseline_samples = baseline_output.audio.shape[-1] if hasattr(baseline_output.audio, "shape") else len(baseline_output.audio)
    baseline_duration = baseline_samples / baseline_output.sample_rate
    if baseline_duration < 10.0:
        raise RuntimeError(f"Expected at least 10 seconds, got {baseline_duration:.2f}")
    print(baseline_output.file_path, f"{baseline_duration:.2f}s")

## 4. Validate a small dataset

Place an authorized JSON Lines manifest at `MANIFEST_PATH`. This cell loads it and resolves relative audio paths from the manifest directory. When the file is absent, template records are used only for an offline preview.

In [ ]:
import json

template_records = [
    {"id": "dia-001", "text": "[S1] Exact authorized transcript.", "audio": str(DATA_ROOT / "session-a.wav"), "speaker_id": "speaker-1", "session_id": "session-a", "consent": True, "license": "owned"},
    {"id": "dia-002", "text": "[S1] Validation uses another session.", "audio": str(DATA_ROOT / "session-b.wav"), "speaker_id": "speaker-1", "session_id": "session-b", "consent": True, "license": "owned"},
]

def load_manifest_records(path):
    loaded = []
    with path.open(encoding="utf-8") as source:
        for line_number, raw_line in enumerate(source, start=1):
            if not raw_line.strip():
                continue
            try:
                record = json.loads(raw_line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON on {path}:{line_number}: {error}") from error
            if not isinstance(record, dict):
                raise TypeError(f"Record {line_number} in {path} must be a JSON object")
            record = dict(record)
            audio = record.get("audio")
            if isinstance(audio, str) and audio.strip():
                audio_path = Path(audio).expanduser()
                if not audio_path.is_absolute():
                    record["audio"] = str((path.parent / audio_path).resolve())
            loaded.append(record)
    if not loaded:
        raise ValueError(f"Manifest contains no records: {path}")
    return loaded

manifest_loaded = MANIFEST_PATH.is_file()
if manifest_loaded:
    records = load_manifest_records(MANIFEST_PATH)
    print(f"Loaded {len(records)} records from {MANIFEST_PATH}")
else:
    records = template_records
    print(f"{MANIFEST_PATH} not found; using template records for offline preview only")

validation_errors = []
seen_ids = set()
for record in records:
    if record["id"] in seen_ids:
        validation_errors.append(f"{record['id']}: duplicate id")
    seen_ids.add(record["id"])
    if not record["text"].strip():
        validation_errors.append(f"{record['id']}: empty transcript")
    if record.get("consent") is not True:
        validation_errors.append(f"{record['id']}: consent missing")
    if not Path(record["audio"]).is_file():
        validation_errors.append(f"{record['id']}: missing audio")
print("ready" if not validation_errors else validation_errors)

In [ ]:
groups = {}
for record in records:
    groups.setdefault(record["session_id"], []).append(record)
if len(groups) < 2:
    raise ValueError("At least two recording sessions are required")
validation_group = sorted(groups)[-1]
train_records = [row for row in records if row["session_id"] != validation_group]
validation_records = [row for row in records if row["session_id"] == validation_group]
assert {row["session_id"] for row in train_records}.isdisjoint(
    {row["session_id"] for row in validation_records}
)
print(len(train_records), len(validation_records))

## 5. Run one optimizer step

Fix every validation error first. Increase `max_steps` only after this smoke run saves and reloads.

In [ ]:
final_artifact = OUTPUT_DIR / "final"
if RUN_TRAINING:
    from voicehub import AutoModelForTextToSpeech
    from voicehub.training.trainer import Trainer

    if validation_errors:
        raise RuntimeError("Fix dataset validation errors before training")
    training_model = AutoModelForTextToSpeech.from_pretrained(
        BASE_MODEL, model_type=MODEL_TYPE, device=DEVICE, lazy_load=True
    )
    training_model.validate_training_support()
    train_dataset = training_model.create_training_dataset(train_records)
    validation_dataset = training_model.create_training_dataset(validation_records)
    arguments = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        max_steps=1,
        per_device_train_batch_size=1,
        learning_rate=5e-5,
        logging_steps=1,
        save_steps=1,
        report_to="none",
        seed=42,
        data_seed=42,
    )
    workflow_trainer = Trainer(
        model=training_model,
        args=arguments,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
    )
    train_output = workflow_trainer.train(resume_from_checkpoint=False)
    workflow_trainer.save_model(final_artifact)
    print(train_output, final_artifact)

## 6. Reload and compare

In [ ]:
if RUN_POST_TRAINING_INFERENCE:
    from voicehub import AutoModelForTextToSpeech, TTSGenerationConfig

    if not final_artifact.is_dir():
        raise FileNotFoundError(final_artifact)
    fine_tuned_model = AutoModelForTextToSpeech.from_pretrained(final_artifact, device=DEVICE)
    fine_tuned_output = fine_tuned_model.generate(
        EVALUATION_TEXT,
        generation_config=TTSGenerationConfig(
            seed=42, max_new_tokens=2048, output_file=OUTPUT_DIR / "fine-tuned.wav"
        ),
    )
    fine_tuned_samples = fine_tuned_output.audio.shape[-1] if hasattr(fine_tuned_output.audio, "shape") else len(fine_tuned_output.audio)
    fine_tuned_duration = fine_tuned_samples / fine_tuned_output.sample_rate
    if fine_tuned_duration < 10.0:
        raise RuntimeError(f"Expected at least 10 seconds, got {fine_tuned_duration:.2f}")
    print(fine_tuned_output.file_path, f"{fine_tuned_duration:.2f}s")

## Next

Use the [training matrix](https://kadirnar.github.io/voicehub/models/training-support/) before adapting this workflow to another model. The [training guide](https://kadirnar.github.io/voicehub/guides/training/) explains objectives, resume artifacts, and exports.